# HidroXAI-MX · entrenamiento de las redes en GPU (Kaggle)

Entrena TCN, ConvNeXt-1D y PatchTST según el protocolo congelado
(`results/xai_benchmark/protocol.md`) con el snapshot v2026.10 del dataset
(Zenodo 10.5281/zenodo.23150556). Proyecto IND-2026-0335 (IPN, PICDT 2026).

**Antes de correr**
1. *Settings* → *Accelerator*: **GPU P100** (o T4); *Internet*: **On**.
2. *Save Version* → **Save & Run All (Commit)**. Corre en segundo plano y guarda la carpeta
   `/kaggle/working/xai_runs` como salida.
3. Si no termina en una sesión (12 h): agrega la salida de la versión anterior como *Input*,
   pon su ruta en `RESUME_FROM` y vuelve a hacer *Save & Run All*. El script salta lo terminado.

Al final, descarga `xai_runs.zip` (pestaña *Output*).

In [ ]:
# Parámetros
REPO = "https://github.com/pantrok/hidroxai-mx.git"
COMMIT = "9d6c5285d6c632499fd5953f749d909ec1ff5e72"             # código fijo con el que se entrena
ZIP_URL = "https://zenodo.org/records/23150556/files/HidroXAI-MX-v2026.10.zip?download=1"
ZIP_MD5 = "6d0cee07231a747aa0ea307c828ab99d"
RESUME_FROM = None                # p. ej. "/kaggle/input/<salida-anterior>/xai_runs"
OUT = "/kaggle/working/xai_runs"
STOP_AFTER_HOURS = 11.0           # deja margen para guardar la salida antes de las 12 h

In [ ]:
import hashlib, os, shutil, subprocess, sys, urllib.request, zipfile
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")
assert torch.cuda.is_available(), "Activa la GPU en Settings → Accelerator"

CODE = "/tmp/hidroxai-mx"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Dataset v2026.10 desde Zenodo: se verifica el MD5 y se extrae solo lo necesario (fuera de la salida)
DATA = "/tmp/hidromx_data"
ZIP = "/tmp/HidroXAI-MX-v2026.10.zip"
if not os.path.exists(f"{DATA}/features/feature_table.parquet"):
    urllib.request.urlretrieve(ZIP_URL, ZIP)
    md5 = hashlib.md5(open(ZIP, "rb").read()).hexdigest()
    assert md5 == ZIP_MD5, f"MD5 distinto: {md5}"
    with zipfile.ZipFile(ZIP) as zf:
        for m in ("features/feature_table.parquet", "processed/estaciones_hidrorivers.csv"):
            zf.extract(m, DATA)
    os.makedirs(f"{DATA}/raw", exist_ok=True)
    os.remove(ZIP)
print(sorted(os.listdir(DATA)))

In [ ]:
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, OUT, dirs_exist_ok=True)
    print("Reanudando desde", RESUME_FROM)
os.makedirs(OUT, exist_ok=True)

In [ ]:
env = {**os.environ, "HIDROXAI_DATA": DATA, "PYTHONPATH": f"{CODE}/src"}
cmd = [sys.executable, "-u", "scripts/xai_03_train_nets.py", "--out", OUT, "--device", "cuda",
       "--stop-after-hours", str(STOP_AFTER_HOURS)]
with subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")
assert p.returncode == 0, f"El script terminó con código {p.returncode}"

In [ ]:
import pandas as pd
man = pd.read_csv(f"{OUT}/compute_manifest.csv")
n_preds = len(os.listdir(f"{OUT}/preds"))
print(f"Corridas terminadas: {n_preds} de 216 · horas de GPU: {man['segundos'].sum() / 3600:.2f}")
shutil.rmtree(f"{OUT}/ckpt", ignore_errors=True) if n_preds == 216 else None
shutil.make_archive("/kaggle/working/xai_runs", "zip", OUT)
print("Salida lista: /kaggle/working/xai_runs.zip")